# UBDS 2026: Basic Python
## Day 3: Take controle over your code


### Topics covered / Сьогоднішні теми
1. Loop comprehention / ...
2. Writing custom functions / ...
3. Reading errors / ...
4. Debugging in Jupyter / ...
5. Good parcticies in Python project structure / ...
6. Introduction to classes / ...


We will use `biopython` package to work with codon table and pairwise sequence aligment. / ...


# Import the tools we need

- `biopython` helps in processing biological sequences / ...
- `logging` helps to handle messeges a program wants user to know about how execution goes / ...
- `warnings` helps to raise custom warnings / ...

In [129]:
from Bio.Data import CodonTable
import logging
import warnings

# PROBLEM 1: How to fit same amount of code into less lines?

### Loop comprehension

To illustraite the principal, we'll do a typical task of bioinformatician work:

Transform DNA sequence into a protein!

In [ ]:
# Load a stadrart codon table
standart_codon_table = CodonTable.unambiguous_dna_by_name["Standard"]

#### Q: What are a triplet and a codon table?

In [20]:
print(standart_codon_table)

Table 1 Standard, SGC0

  |  T      |  C      |  A      |  G      |
--+---------+---------+---------+---------+--
T | TTT F   | TCT S   | TAT Y   | TGT C   | T
T | TTC F   | TCC S   | TAC Y   | TGC C   | C
T | TTA L   | TCA S   | TAA Stop| TGA Stop| A
T | TTG L(s)| TCG S   | TAG Stop| TGG W   | G
--+---------+---------+---------+---------+--
C | CTT L   | CCT P   | CAT H   | CGT R   | T
C | CTC L   | CCC P   | CAC H   | CGC R   | C
C | CTA L   | CCA P   | CAA Q   | CGA R   | A
C | CTG L(s)| CCG P   | CAG Q   | CGG R   | G
--+---------+---------+---------+---------+--
A | ATT I   | ACT T   | AAT N   | AGT S   | T
A | ATC I   | ACC T   | AAC N   | AGC S   | C
A | ATA I   | ACA T   | AAA K   | AGA R   | A
A | ATG M(s)| ACG T   | AAG K   | AGG R   | G
--+---------+---------+---------+---------+--
G | GTT V   | GCT A   | GAT D   | GGT G   | T
G | GTC V   | GCC A   | GAC D   | GGC G   | C
G | GTA V   | GCA A   | GAA E   | GGA G   | A
G | GTG V   | GCG A   | GAG E   | GGG G   | G
--+---------

In [36]:
# Set a DNA sequence, which encodes a short protein
dna_sequence = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"

# Initialise a varibale to store protein sequence
protein1 = ""

# Loop through the sequence in steps of 3
for i in range(0, len(dna_sequence), 3):

    # Select one consecutive codon
    codon = dna_sequence[i:i+3]

    # Retreave the amino acid encoded by the codon. If codon is atypical - insert "?"
    # Add amino acid to a protein string
    protein1 += standart_codon_table.forward_table.get(codon, "?")

In [37]:
protein2 = "".join([standart_codon_table.forward_table.get(dna_sequence[i:i+3], "?") for i in range(0, len(dna_sequence), 3)])

#### Q: Are protein1 and protein2 different?

In [44]:
print("Protein 1", protein1)
print("Protein 2", protein2)
print("Are they the same?", protein1==protein2)

Protein 1 MAIVMGR?KGAR?
Protein 2 MAIVMGR?KGAR?
Are they the same? True


**Let's unfold what just happend.**

It is possible to write a loop in one line. 

- Assign a list to a variable.

```python
protein2 = []
```

- Include "for" statement.

```python
protein2 = [for i in range(0, len(dna_sequence), 3)]
```

- Add execution of the function before "for" statement. Note, how instead of creating `codon` variable, its value is passed directly.

```python
protein2 = [
    standart_codon_table.forward_table.get(dna_sequence[i:i+3], "?") 
    for i in range(0, len(dna_sequence), 3)
    ]
```

- For our case we have to add another treak. Now, `protein2` is a list of amino acids. To convert it into the string, pass the whole statement to the function `join()` with delimeter of empty string "". This concatenates all string elements of the list.

```python
protein2 = "".join([...])
```

_Offcourse you do not have to follow the exact order of the steps. But the final result always looks the same._

Something does not make sence. 

Why this sequence containes unrecognised triplets? Maybe the DNA sequence is not devidable by 3?

We can incorporate a condition inside a loop comprehention to damand that all checked sequences are triplets.

A single condition always follows for statement.

In [ ]:
# Here we again unfold loop comprehention for clearity, but it can be writen in one line
protein3 = "".join(
    [
        standart_codon_table.forward_table.get(dna_sequence[i:i+3], "?") 
        for i in range(0, len(dna_sequence), 3)
        if len(dna_sequence[i:i+3]) == 3 
    ]
)

#### Q: How added condition makes protein3 different from protein2?

In [ ]:
print("Protein 2 (w/o condition) length: ", len(protein2))
print("Protein 3 (w/ condition) length: ", len(protein3))
print(protein2)
print(protein3)

Protein2 (w/o condition) length:  13
Protein (w/ condition) length:  12
MAIVMGR?KGAR?
MAIVMGR?KGAR


There is still one unrecognised triplet in the sequence.

#### Q: Does codon table include stop codons?

In [42]:
stop_codon = "TGA"
print(standart_codon_table.forward_table.get(stop_codon, "?"))

?


We can add multiple conditions. 

As in full loops, loop conprehention condition can not only filter out the elements, but also control an output.

Wrap the function call statement into `()` and insert a condition. Whatever has to be executed if the condition is fulfilled goes before `if` and second statement goes after `else`. 


In [ ]:
protein4 = "".join(
    [
        (
            standart_codon_table.forward_table.get(dna_sequence[i:i+3], "?")
            if dna_sequence[i:i+3] in standart_codon_table.stop_codons
            else
            "*"
        )
        for i in range(0, len(dna_sequence), 3)
        if len(dna_sequence[i:i+3]) == 3 
    ]
)

In [47]:
print("Protein 3 (w/ one condition) length: ", len(protein3))
print("Protein 4 (w/ two conditions) length: ", len(protein4))
print(protein3)
print(protein4)

Protein 3 (w/ one condition) length:  12
Protein 4 (w/ two conditions) length:  12
MAIVMGR?KGAR
MAIVMGR*KGAR


From biological perspective, Protein 4 does not make sence. The protein sequence have to stop after stop codon.

Note, you can also put several conditions one after another, where second statement going after first `else` is executed, when the second `if` is fullfiled.

#### Q: Can we put a condition for a loop to stop inside comprehention?

In [ ]:
# This nested condition tries to stop the loop when stop codon is met
protein5 = "".join(
    [
        (
            standart_codon_table.forward_table.get(dna_sequence[i:i+3], "?")
            if dna_sequence[i:i+3] not in standart_codon_table.stop_codons
            else 
            break 
            if dna_sequence[i:i+3] in standart_codon_table.stop_codons
            else
            ""
        )
        for i in range(0, len(dna_sequence), 3)
        if len(dna_sequence[i:i+3]) == 3 
    ]
)

SyntaxError: invalid syntax (3059466520.py, line 8)

It is impossible to stop a loop comprehention untill it run out of things to loop through. Also, note how we had to write `dna_sequence[i:i+3]` four times.

Loop comprehension is ment to shrink simple tasks in one line of code.

To accomodate more complex tasks, either use full loops or write a custom function.

# PROBLEM 2: I cannot find a package that does what I want.

Similarly to how `biopython` functions work for general bioinformatics purposes, we can create our own functions that suite the specific needs of our analysis.

Another advantage of writing a function - you do not need to repeat the same big chunk of code anymore. 

Lets create a function that performs codon->amino acid conversion and is aware of triplet condition and stop codons. Meet a custom function `real_translation`!

#### Q: What do we want a custom function to take as input and give as output?

Similar to variables, a custom function has to be defined before execution.

In [ ]:
protein5 = "".join([real_translation(dna_sequence[i:i+3]) for i in range(0, len(dna_sequence), 3)])

Custom functions can take nothing as input, one or more arguments.

They can return or not return objects.

In [ ]:
# It is a good practice to describe inside the function what it does
def real_translation(dna_sequence):
    """
    This function translates dna_sequence in real protein sequence.
    Input:
        dna_sequence: string of nucleotides
    Output:
        protein: string of amino acids
    """
    protein = ""
    # Loop through the sequence in steps of 3
    for i in range(0, len(dna_sequence), 3):
        codon = dna_sequence[i:i+3]
        
        # Make sure codon has length 3
        if len(codon) < 3:
            break
        # Stop when a stop codon was found
        if codon in standart_codon_table.stop_codons:
            break
        else:
            # Retreave the amino acid encoded by the codon. If codon is atypical - insert "?"
            protein += standart_codon_table.forward_table.get(codon, "?")
    

Attempt N 1

In [74]:
dna_sequence = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"
real_translation()
print(protein)

TypeError: real_translation() missing 1 required positional argument: 'dna_sequence'

If a custom function was designed to recieve arguments - it will damand them.

Attempt N 2

In [ ]:
real_translation(dna_sequence)
print(protein)

NameError: name 'protein' is not defined

If a variable was defined inside a custom function - it is not accessible from outside.

To make a variable accessible, add this to the end of the function `real_translation`.

```python
return protein
```

Note, how below we definde a target variable (`protein`) as a storage of `real_translation` output.

In [84]:
dna_sequence = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"
protein = real_translation(dna_sequence)
print(protein)

MAIVMGR


Let's say that you want to use a different codon table for different parts of your analysis. **Usually** you use a standart one, but sometimes - of Mitochondria.

In [89]:
# Find codon table of any mitochonria
for id, table in CodonTable.unambiguous_dna_by_id.items():
    if "mitochondria" in table.names[0].lower():
        print(id, "-", table.names)

2 - ['Vertebrate Mitochondrial', 'SGC1']
3 - ['Yeast Mitochondrial', 'SGC2']
4 - ['Mold Mitochondrial', 'Protozoan Mitochondrial', 'Coelenterate Mitochondrial', 'Mycoplasma', 'Spiroplasma', 'SGC3']
5 - ['Invertebrate Mitochondrial', 'SGC4']
9 - ['Echinoderm Mitochondrial', 'Flatworm Mitochondrial', 'SGC8']
13 - ['Ascidian Mitochondrial', None]
14 - ['Alternative Flatworm Mitochondrial', None]
16 - ['Chlorophycean Mitochondrial', None]
21 - ['Trematode Mitochondrial', None]
22 - ['Scenedesmus obliquus Mitochondrial', None]
23 - ['Thraustochytrium Mitochondrial', None]
24 - ['Pterobranchia Mitochondrial', None]
33 - ['Cephalodiscidae Mitochondrial', None]


#### Q: Are mitochonria codon table and standart codon table different?

In [88]:
CodonTable.unambiguous_dna_by_name["Standard"] == CodonTable.unambiguous_dna_by_name["Vertebrate Mitochondrial"]

False

To make our function appliable to any organism, we could pass a `codon_table_name` as argument and it will be retrieved automaticaly.

What if we want a standart codon table to be used by default?

To do so, give a value to the argument directly in function definition.
In this case the user does not have to pass `codon_table_name="Standart"` argument, only if a different from "Standart" table is requered.

In [105]:
def real_translation(dna_sequence, codon_table_name="Standard"):
    """
    This function translates dna_sequence in real protein sequence.
    Input:
        dna_sequence: string of nucleotides
        codon_table_name: string name
    Output:
        protein: string of amino acids
    """
    # Retreive a codon table
    codon_table = CodonTable.unambiguous_dna_by_name[codon_table_name]
    protein = ""
    # Loop through the sequence in steps of 3
    for i in range(0, len(dna_sequence), 3):
        codon = dna_sequence[i:i+3]
        
        # Make sure codon has length 3
        if len(codon) < 3:
            break
        # Stop when a stop codon was found
        if codon in codon_table.stop_codons:
            break
        else:
            # Retreave the amino acid encoded by the codon. If codon is atypical - insert "?"
            protein += codon_table.forward_table.get(codon, "?")
    return protein

We can also define an argument by calling its name.

Note, keyword arguments (with default value) have to be always called after positional arguments (without default).

In [93]:
dna_sequence = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"
protein = real_translation(codon_table_name = "Vertebrate Mitochondrial", dna_sequence)
print(protein)

SyntaxError: positional argument follows keyword argument (4158634327.py, line 2)

#### Q: Does our function finaly work?

It is a good practice to test a custom function on various scenarious. This way you will know if it works as expected.

Test N 1 normal dna sequence

In [80]:
dna_sequence = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"
protein = real_translation(dna_sequence)
print(protein)

MAIVMGR


Test N 2

In [106]:
dna_sequence = ["ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"]
protein = real_translation(dna_sequence)
print(protein)

Test N 3

In [82]:
dna_sequence = "AUGGCCAUUGUAAUGGGCCGCUGAAAGGGUGCCCGUA"
protein = real_translation(dna_sequence)
print(protein)

?A???GR?K?A?


Test N 4

In [81]:
dna_sequence = "AT"
protein = real_translation(dna_sequence)
print(protein)

Test N 5

In [116]:
dna_sequence = "GGGCTAGCCATTGTAATGGGCCGCAAGGGTGCCCGTA"
protein = real_translation(dna_sequence)
print(protein)

GLAIVMGRKGAR


Test N 6

In [111]:
dna_sequence = "TGATAA"
protein = real_translation(dna_sequence)
print(protein)

Test N 7

In [ ]:
dna_sequence = "GGGCTAATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"
codon_table_name = "Dragon"
protein = real_translation(dna_sequence, codon_table_name)
print(protein)

We identified several problematic cases:
- input is not a string
- non-DNA sequence
- non-triplet sequence
- sequence starting with either no start or stop codon
- sequence contains only non-coding triplets
- unknown codon table name

To avoid this cases, we can worn a user (you or your collegues) about weird input or program behaviour.

What code can communicate to us?

**INFO** -> "What is the program doing overall?" (for casual usage)

**DEBUG** -> "What exactly is happening step-by-step?" (for developers)

**WARNING** -> "Signal something is not ideal." (program continues)

**EXCEPTION** -> "Signal something is wrong, but handleable." (program stops, you can correct)

**ERROR** -> "Signal of code or system issues." (program stops, you cannot correct, unless you are a developer)


Info and Debug messeges can be simpli writen as print statements.

Logging is a good practice, when you develop a script, an tool or a pipeline for internal usage.

In [122]:
# set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)

This function looks huge, but only because it tries to be both user- and developer-freindly.

In [134]:
def real_translation(dna_sequence, codon_table_name="Standard"):
    """
    This function translates dna_sequence in real protein sequence.
    Input:
        dna_sequence: string of nucleotides
        codon_table_name: string name
    Output:
        protein: string of amino acids
    """
    logging.info(f"Start execution of real_translation()")

    # Check codon_table_name value
    logging.debug(f"Use codon table {codon_table_name}")
    table_names = list(CodonTable.unambiguous_dna_by_name.keys())
    if codon_table_name not in table_names:
        logging.exception(f"Codon table name is invalid. Try one of these: {table_names}")
        raise ValueError
    
    # Retreive a codon table
    codon_table = CodonTable.unambiguous_dna_by_name[codon_table_name]
    logging.info(f"Successfully retrieved codon table.")

    # Check dna_sequence type
    logging.debug(f"Type of dna_sequence argument {type(dna_sequence)}")
    if not isinstance(dna_sequence, str):
        logging.exception(f"Argument dna_sequence type invalid. It should be a string (str).")
        raise ValueError
    
    # Check dna_sequence alphabet
    expected_alphabet = set("ATGC")
    sequence_alphabet = set(dna_sequence)
    logging.debug(f"String dna_sequence contains letters {sequence_alphabet}")
    if expected_alphabet != sequence_alphabet:
        logging.warning(f"Function real_translation() is suboptimal for non-DNA sequences. Use {expected_alphabet} alphabet.")
        warnings.warn(f"Non-DNA sequence was given.")
    
    # Check if dna_sequence containes at least one codon
    logging.debug(f"String dna_sequence legth {len(dna_sequence)}")
    if len(dna_sequence)<3:
        logging.exception(f"String dna_sequence is shorter than one codon. Minimal acceptable length is 3.")
        raise ValueError


    protein = ""
    codons = []
    cdna = 0 ; real_stop_codon = False
    # Loop through the sequence in steps of 3
    for i in range(0, len(dna_sequence), 3):

        codon = dna_sequence[i:i+3]

        # Make sure codon has length 3
        if len(codon) < 3:
            break

        # Check that sequence is coding from the start
        if i==0:
            print(codon)
            cdna = 1 if codon == "ATG" else 0
            if not cdna:
                logging.warning(f"String dna_sequence starts as non-coding. Skipping non-coding triplets.")
                warnings.warn(f"Non-coding sequence is part of input.")
                continue
        
        # Check that start codon was found
        if codon == "ATG":
            cdna +=1
            if cdna==1:
                logging.info(f"Start codon was found on position {i}.")
        
        if cdna>1 and not real_stop_codon:
            real_stop_codon = codon in codon_table.stop_codons
            if real_stop_codon:
                logging.info(f"Stop codon was found on position {i}.")

        
        
        
        # Recod unique codons
        if codon not in codons:
            codons.append(codon)

        # Stop when a stop codon was found
        if codon in codon_table.stop_codons:
            break
        else:
            # Retreave the amino acid encoded by the codon. If codon is atypical - insert "?"
            protein += codon_table.forward_table.get(codon, "?")

    if protein=="":
        logging.debug(f"String dna_sequence unique codons {codons}")
        logging.warning(f"Protein sequence is empty. Probably dna_sequence consisted only of non-coding triplets.")
        warnings.warn(f"Non-coding sequence was given.")
    elif protein!="" and not real_stop_codon:
        logging.exception(f"Stop codon was never found. Corrupted coding sequence")
        raise ValueError

    return protein

In [135]:
dna_sequence = "AAAATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGTA"
protein = real_translation(dna_sequence)
print(protein)

INFO:root:Start execution of real_translation()
INFO:root:Successfully retrieved codon table.
C:\Users\vvsha\AppData\Local\Temp\ipykernel_90684\3797548731.py:62: UserWarning: Non-coding sequence is part of input.
  warnings.warn(f"Non-coding sequence is part of input.")
INFO:root:Start codon was found on position 3.
INFO:root:Stop codon was found on position 24.


AAA
MAIVMGR


In [ ]:
import logging
from Bio.Data import CodonTable



def translate_codon(codon):
    table = CodonTable.unambiguous_dna_by_id[1]
    
    if codon in table.stop_codons:
        logging.info(f"Stop codon encountered: {codon}")
        return "*"
    
    return table.forward_table.get(codon, "?")

translate_codon("TGA")

INFO:root:Stop codon encountered: TGA


'*'

# EXCERSISE 1

Lets make real_translation() even more suffisticated.
frame